# Comparação 1 — `backlash.py` original

Este notebook usa a classe original da dissertação. O objetivo é verificar se a atribuição `gear_mesh_stiffness = 0` remove efetivamente a matriz linear de acoplamento do `MultiRotor` atual.

## 1. Imports e carregamento da classe original

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ross as rs

ROOT = Path(r'C:\Users\M\Desktop\doutorado\teste_backlash_ross')
BASE_BACKLASH = ROOT / 'mestrado' / 'backlash.py'
spec = spec_from_file_location('backlash_original_comparison', BASE_BACKLASH)
module = module_from_spec(spec)
spec.loader.exec_module(module)
Backlash = module.Backlash
print(f'Classe carregada de: {BASE_BACKLASH}')

## 2. Modelagem do multirrotor

In [ ]:
def build_multirotor():
    z = 20
    module = 0.01
    steel = rs.Material(name='Steel', rho=7850, E=2e11, Poisson=0.3)
    stiff = rs.Material(name='Steel_Stiff', rho=0.01, E=1e15, Poisson=0.3)
    shaft = [rs.ShaftElement(L=0.0001, idl=0.0, odl=0.0001, material=stiff, n=0)]
    bearing = rs.BearingElement(n=0, kxx=1e8, kyy=1e8, cxx=512.64, cyy=512.64)
    pd_gear = module * z
    bore = np.sqrt(pd_gear**2 - 4*6.57/(np.pi*0.03*steel.rho))
    gear = rs.GearElementTVMS(n=0, material=steel, width=0.03, bore_diameter=bore,
        module=module, n_teeth=z, pr_angle=np.radians(20), helix_angle=0,
        addendum_coeff=1, tip_clearance_coeff=0.25)
    gear.m = 6.57; gear.Ip = 0.0365; gear.Id = 0.0001*0.0365/2
    rotor = rs.Rotor(shaft_elements=shaft, disk_elements=[gear], bearing_elements=[bearing])
    return rs.MultiRotor(driving_rotor=rotor, driven_rotor=rotor, coupled_nodes=(0,0),
        update_mesh_stiffness=True, square_varying_stiffness={'enable': True, 'amplitude_ratio': 0.275},
        orientation_angle=0.0, position='above')

multirotor = build_multirotor()
omega = 4500*np.pi/30
backlash = Backlash(multirotor, omega, b0=50e-6, error_amp=20e-6,
    gear_mesh_stiffness=None, num_points_cicle=10, n_cicles=1, cut_cicles=0,
    use_multirotor_coupling_stiffness=False)
model = backlash.multirotor
print('backlash original ativado; use_multirotor_coupling_stiffness=False')

## 3. Extração da matriz nos DOFs das engrenagens

In [ ]:
gear_dofs = list(model.mesh.driving_gear.dof_global_index.values()) + list(model.mesh.driven_gear.dof_global_index.values())
K = model.K(omega)
K0 = model._join_matrices(model.rotors['driving'].K(omega, omega), model.rotors['driven'].K(omega, omega*model.mesh.gear_ratio))
Kgear = K[np.ix_(gear_dofs, gear_dofs)]
K0gear = K0[np.ix_(gear_dofs, gear_dofs)]
Kcoupling = model.K_coupling * model.mesh.stiffness
Kcoupling_gear = Kcoupling

print('DOFs globais das engrenagens:', gear_dofs)
print('mesh.stiffness =', model.mesh.stiffness)
print('norma Kgear-K0gear =', np.linalg.norm(Kgear-K0gear))
print('norma coupling esperado =', np.linalg.norm(Kcoupling_gear))

## 4. Valores das matrizes e teste direto

In [ ]:
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 240)
print('Kgear:')
display(pd.DataFrame(Kgear, index=gear_dofs, columns=gear_dofs))
print('K0gear:')
display(pd.DataFrame(K0gear, index=gear_dofs, columns=gear_dofs))
print('Kgear - K0gear:')
display(pd.DataFrame(Kgear-K0gear, index=gear_dofs, columns=gear_dofs))
np.testing.assert_allclose(Kgear-K0gear, Kcoupling_gear, rtol=1e-12, atol=1e-6)
print('Resultado: a contribuição K_coupling * mesh.stiffness está presente.')

## 5. Visualização da contribuição adicionada

In [ ]:
fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(Kgear-K0gear, cmap='coolwarm')
ax.set_title('backlash.py original: Kgear - K0gear')
ax.set_xlabel('DOF da engrenagem'); ax.set_ylabel('DOF da engrenagem')
fig.colorbar(im, ax=ax, label='Rigidez')
plt.show()